# Paper tables (main text and supplementary material)

This notebook builds every table of the paper from the CSV files written by
`pairwise_analysis`, `clustering_analysis`, `seed_clustering_analysis`,
`elsst_diagnostic_analysis` and `apa_diagnostic_analysis`. It computes nothing new.

Each table is saved twice in `analysis/results/paper/`: as CSV and as a LaTeX table (`.tex`, booktabs).
Main-text tables start with `table`, supplementary tables with `S`.

Run it after both diagnostic notebooks.

## 0. Setup

In [1]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "analysis").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing scripts/ and analysis/")

sys.path.insert(0, str(_find_project_root(Path.cwd().resolve())))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display, Markdown

from scripts import config

OUT_DIR = config.PROJECT_ROOT / "analysis/results/paper"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAMES = {"elsst": "ELSST", "apa": "APA"}
TECHNIQUE_ORDER = ["Clustering", "Pairwise", "Seeded"]
MODEL_ORDER = [config.MODELS[m]["display_name"] for m in config.MODELS] + [config.BASELINE_DISPLAY_NAME]

# Model keys (Hugging Face ids) to display names, for the calibration files
KEY_TO_NAME = {key: info["display_name"] for key, info in config.MODELS.items()}
KEY_TO_NAME[config.BASELINE_KEY] = config.BASELINE_DISPLAY_NAME

# Both diagnostic notebooks must have been run with the current code
# (seeded_draws.csv only exists in the current version of the results).
for key, name in DATASET_NAMES.items():
    if not (config.RESULTS_DIRS[key] / "seeded_draws.csv").exists():
        raise FileNotFoundError(f"The {name} results are missing or from an older version. "
                                f"Run {key}_diagnostic_analysis.ipynb first.")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("Tables are written to", OUT_DIR)

Tables are written to /home/rass/Desktop/projects/Social_Science_Construct_Harmonization/analysis/results/paper


In [2]:
# Helpers: read the result files of both data sets, order rows, save CSV and LaTeX.

def read_results(file_name):
    """One CSV from each data set, stacked, with a 'dataset' column (ELSST first)."""
    frames = []
    for key, name in DATASET_NAMES.items():
        path = config.RESULTS_DIRS[key] / file_name
        if not path.exists():
            raise FileNotFoundError(f"{path} is missing. Run the {name} diagnostic notebook first.")
        df = pd.read_csv(path)
        df.insert(0, "dataset", name)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def tidy(df):
    """Capitalised technique names, rows in the order data set > technique > model."""
    df = df.copy()
    if "technique" in df:
        df["technique"] = df["technique"].str.capitalize()
        df["technique"] = pd.Categorical(df["technique"], TECHNIQUE_ORDER, ordered=True)
    if "model" in df:
        df["model"] = pd.Categorical(df["model"], MODEL_ORDER, ordered=True)
    if "dataset" in df:
        df["dataset"] = pd.Categorical(df["dataset"], list(DATASET_NAMES.values()), ordered=True)
    sort_cols = [col for col in ["dataset", "technique", "model"] if col in df]
    df = df.sort_values(sort_cols).reset_index(drop=True)
    for col in ["dataset", "technique", "model"]:
        if col in df:
            df[col] = df[col].astype(str)
    return df


def latex_escape(text):
    for old, new in [("\\", "\\textbackslash{}"), ("&", "\\&"), ("%", "\\%"), ("_", "\\_"), ("#", "\\#")]:
        text = text.replace(old, new)
    return text


LABEL_COLUMNS = {"Dataset", "Technique", "Model", "Split", "Pipeline A", "Pipeline B",
                 "Selected parameters", "Overlap band", "Variation"}


def to_latex(df, caption, label):
    """A simple booktabs table. Numbers are written as they appear in df (format them first)."""
    align = "".join("l" if col in LABEL_COLUMNS else "r" for col in df.columns)
    lines = ["\\begin{table}[ht]", "\\centering", "\\small",
             f"\\caption{{{latex_escape(caption)}}}", f"\\label{{{label}}}",
             f"\\begin{{tabular}}{{{align}}}", "\\toprule",
             " & ".join(latex_escape(str(col)) for col in df.columns) + " \\\\", "\\midrule"]
    for _, row in df.iterrows():
        lines.append(" & ".join(latex_escape(str(v)) for v in row.values) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines) + "\n"


def save_table(df, name, caption):
    """Save as CSV and LaTeX, and show it."""
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    (OUT_DIR / f"{name}.tex").write_text(to_latex(df, caption, f"tab:{name}"))
    display(Markdown(f"**{name}.** {caption}"))
    display(df)


def exact_rate(rate, n):
    """The diagnostic CSVs store some rates rounded to 4 decimals. A rate is a count divided by n,
    so the count is recovered exactly (round(rate * n)) and the rate recomputed without rounding."""
    return np.round(rate * n) / n


def fmt(df, decimals=3, columns=None):
    """Round numeric columns to strings with a fixed number of decimals (empty for missing values)."""
    df = df.copy()
    for col in (columns or df.select_dtypes("number").columns):
        df[col] = df[col].map(lambda v: "" if pd.isna(v) else f"{v:.{decimals}f}")
    return df

---
## 1. Main-text tables

### Table 1. Primary performance

In [3]:
perf = tidy(read_results("primary_performance.csv"))
ci = tidy(read_results("bootstrap_intervals.csv"))
ci = ci[ci["metric"] == "f1"][["dataset", "technique", "model", "ci_low", "ci_high"]]
perf = perf.merge(ci, on=["dataset", "technique", "model"], how="left")

table1 = pd.DataFrame({
    "Dataset": perf["dataset"],
    "Technique": perf["technique"],
    "Model": perf["model"],
    "Precision": perf["precision"].map("{:.3f}".format),
    "Recall": perf["recall"].map("{:.3f}".format),
    "F1 [95% CI]": [f"{f:.3f} [{lo:.3f}, {hi:.3f}]" for f, lo, hi in zip(perf["f1"], perf["ci_low"], perf["ci_high"])],
    "FP per 10,000": perf["fp_per_10k_negatives"].map("{:.2f}".format),
    "F1 at ELSST prevalence": perf["f1_at_reference_prevalence"].map("{:.3f}".format),
})
save_table(table1, "table1_primary_performance",
           "Precision, recall and F1 by data set, technique and model. 95% intervals from 1,000 concept-level "
           "bootstrap samples (seeded clustering: mean over 10 seed draws, interval from the median draw). "
           "FP per 10,000: false positives per 10,000 non-synonym pairs. F1 at ELSST prevalence: precision "
           "recomputed with the class balance of the ELSST test split, recall and false-positive rate unchanged.")

**table1_primary_performance.** Precision, recall and F1 by data set, technique and model. 95% intervals from 1,000 concept-level bootstrap samples (seeded clustering: mean over 10 seed draws, interval from the median draw). FP per 10,000: false positives per 10,000 non-synonym pairs. F1 at ELSST prevalence: precision recomputed with the class balance of the ELSST test split, recall and false-positive rate unchanged.

,Dataset,Technique,Model,Precision,Recall,F1 [95% CI],"FP per 10,000",F1 at ELSST prevalence
0,ELSST,Clustering,All-MPNet-Base-v2,0.513,0.503,"0.508 [0.469, 0.549]",4.23,0.508
1,ELSST,Clustering,MPNet-Personality,0.481,0.492,"0.487 [0.441, 0.527]",4.70,0.487
2,ELSST,Clustering,SciBERT (SciVocab),0.372,0.292,"0.327 [0.288, 0.370]",4.38,0.327
3,ELSST,Clustering,BERT Base,0.290,0.283,"0.286 [0.248, 0.326]",6.17,0.286
4,ELSST,Pairwise,All-MPNet-Base-v2,0.695,0.370,"0.483 [0.438, 0.525]",1.44,0.483
5,ELSST,Pairwise,MPNet-Personality,0.653,0.347,"0.453 [0.412, 0.494]",1.64,0.453
6,ELSST,Pairwise,SciBERT (SciVocab),0.354,0.220,"0.272 [0.237, 0.306]",3.57,0.272
7,ELSST,Pairwise,BERT Base,0.273,0.114,"0.161 [0.133, 0.191]",2.68,0.161
8,ELSST,Pairwise,Character 3-gram (baseline),0.394,0.137,"0.203 [0.174, 0.239]",1.86,0.203
9,ELSST,Seeded,All-MPNet-Base-v2,0.685,0.336,"0.451 [0.400, 0.506]",1.37,0.451


### Table 2. Lexical trap test

In [4]:
trap = tidy(read_results("lexical_trap.csv"))
# Rates from the false-positive counts (no rounding)
fpr_hard = trap["fp_hard"] / trap["n_hard"]
fpr_easy = trap["fp_easy"] / trap["n_easy"]
table2 = pd.DataFrame({
    "Dataset": trap["dataset"], "Technique": trap["technique"], "Model": trap["model"],
    "FPR (hard)": fpr_hard.map("{:.3f}".format),
    "FP per 10,000 easy": (10_000 * fpr_easy).map("{:.2f}".format),
    "Hard / easy FPR ratio": (fpr_hard / fpr_easy).map(lambda v: "" if not np.isfinite(v) else f"{v:,.0f}"),
})
n_hard = trap.groupby("dataset")["n_hard"].first()
n_easy = trap.groupby("dataset")["n_easy"].first()
save_table(table2, "table2_lexical_trap",
           "Lexical trap test. False-positive rate on hard negatives (character 3-gram Jaccard > 0.5) and false "
           "positives per 10,000 easy negatives (Jaccard = 0). N hard = "
           f"{n_hard['ELSST']:,} on ELSST and {n_hard['APA']:,} on APA; N easy = {n_easy['ELSST']:,} and "
           f"{n_easy['APA']:,}. The 3-gram baseline flags every hard negative by construction (its threshold is "
           "below 0.5) and is shown for reference only.")

**table2_lexical_trap.** Lexical trap test. False-positive rate on hard negatives (character 3-gram Jaccard > 0.5) and false positives per 10,000 easy negatives (Jaccard = 0). N hard = 137 on ELSST and 3,018 on APA; N easy = 1,454,151 and 49,852,643. The 3-gram baseline flags every hard negative by construction (its threshold is below 0.5) and is shown for reference only.

,Dataset,Technique,Model,FPR (hard),"FP per 10,000 easy",Hard / easy FPR ratio
0,ELSST,Clustering,All-MPNet-Base-v2,0.204,1.66,"1,233"
1,ELSST,Clustering,MPNet-Personality,0.270,1.42,"1,906"
2,ELSST,Clustering,SciBERT (SciVocab),0.109,1.81,605
3,ELSST,Clustering,BERT Base,0.248,3.49,712
4,ELSST,Pairwise,All-MPNet-Base-v2,0.248,0.14,"17,185"
5,ELSST,Pairwise,MPNet-Personality,0.226,0.12,"18,280"
6,ELSST,Pairwise,SciBERT (SciVocab),0.124,1.42,876
7,ELSST,Pairwise,BERT Base,0.095,2.50,380
8,ELSST,Pairwise,Character 3-gram (baseline),1.000,0.00,
9,ELSST,Seeded,All-MPNet-Base-v2,0.204,0.16,"12,922"


### Table 3. Semantic decay test

In [5]:
decay = tidy(read_results("semantic_decay.csv"))
rec_easy = exact_rate(decay["recall_easy"], decay["n_easy"])
rec_hard = exact_rate(decay["recall_hard"], decay["n_hard"])
table3 = pd.DataFrame({
    "Dataset": decay["dataset"], "Technique": decay["technique"], "Model": decay["model"],
    "Recall (easy)": rec_easy.map("{:.3f}".format), "Recall (hard)": rec_hard.map("{:.3f}".format),
    "Delta recall": (rec_hard - rec_easy).map("{:+.3f}".format),
    "Retention": (rec_hard / rec_easy).map(lambda v: "" if not np.isfinite(v) else f"{v:.3f}"),
})
n_easy = decay.groupby("dataset")["n_easy"].first()
n_hard = decay.groupby("dataset")["n_hard"].first()
save_table(table3, "table3_semantic_decay",
           "Semantic decay test. Recall on easy positives (Jaccard > 0.5) and hard positives (Jaccard = 0). "
           "Delta recall = recall (hard) - recall (easy); retention = recall (hard) / recall (easy). "
           f"N easy / hard = {n_easy['ELSST']:,} / {n_hard['ELSST']:,} on ELSST and "
           f"{n_easy['APA']:,} / {n_hard['APA']:,} on APA.")

**table3_semantic_decay.** Semantic decay test. Recall on easy positives (Jaccard > 0.5) and hard positives (Jaccard = 0). Delta recall = recall (hard) - recall (easy); retention = recall (hard) / recall (easy). N easy / hard = 152 / 517 on ELSST and 1,086 / 1,447 on APA.

,Dataset,Technique,Model,Recall (easy),Recall (hard),Delta recall,Retention
0,ELSST,Clustering,All-MPNet-Base-v2,0.868,0.271,-0.598,0.312
1,ELSST,Clustering,MPNet-Personality,0.882,0.250,-0.632,0.283
2,ELSST,Clustering,SciBERT (SciVocab),0.645,0.060,-0.585,0.093
3,ELSST,Clustering,BERT Base,0.605,0.081,-0.524,0.134
4,ELSST,Pairwise,All-MPNet-Base-v2,0.862,0.091,-0.771,0.105
5,ELSST,Pairwise,MPNet-Personality,0.855,0.072,-0.784,0.084
6,ELSST,Pairwise,SciBERT (SciVocab),0.533,0.025,-0.508,0.047
7,ELSST,Pairwise,BERT Base,0.336,0.037,-0.299,0.110
8,ELSST,Pairwise,Character 3-gram (baseline),1.000,0.000,-1.000,0.000
9,ELSST,Seeded,All-MPNet-Base-v2,0.789,0.089,-0.700,0.113


### Table 4. Embedding stability test

In [6]:
stab = tidy(read_results("stability.csv"))
base = stab[stab["noise_level"] == 0.0].set_index(["dataset", "technique", "model"])
top = stab["noise_level"].max()
high = stab[stab["noise_level"] == top].set_index(["dataset", "technique", "model"])
is_seeded = base.index.get_level_values("technique") == "Seeded"

# The reference run is the noise-free run (seeded clustering: seed draw 0). Delta F1 is the mean change from it,
# so its F1 is mean F1 at the top noise level minus the mean change.
table4 = pd.DataFrame({
    "F1 reference run": high["mean_f1"] - high["mean_delta_f1"],
    f"F1 (sigma = {top:.2f})": high["mean_f1"],
    f"SD (sigma = {top:.2f})": high["sd_f1"],
    f"Delta F1 (sigma = {top:.2f})": high["mean_delta_f1"],
    f"Agreement (sigma = {top:.2f})": high["mean_agreement"],
    "SD over seed draws": base["sd_f1"].where(is_seeded),
    "Agreement between seed draws": base["mean_agreement"].where(is_seeded),
}).reset_index()
table4 = tidy(table4).rename(columns={"dataset": "Dataset", "technique": "Technique", "model": "Model"})
table4 = fmt(table4)
save_table(table4, "table4_embedding_stability",
           f"Embedding stability test. Gaussian noise added to the embeddings, {config.N_RELIABILITY_RUNS} runs per "
           "noise level. Reference run: the noise-free run (seeded clustering: seed draw 0, so the noise runs keep "
           "the seeds fixed). Delta F1: mean change from the reference run. Agreement: Jaccard overlap between the "
           "predicted synonym pairs of a noisy run and of the reference run. Seed draws (seeded clustering only, "
           f"no noise, {config.N_SEED_DRAWS} draws): SD of F1, and mean agreement of draws 1 to "
           f"{config.N_SEED_DRAWS - 1} with draw 0. All noise levels are in Supplementary Table S3.")

**table4_embedding_stability.** Embedding stability test. Gaussian noise added to the embeddings, 10 runs per noise level. Reference run: the noise-free run (seeded clustering: seed draw 0, so the noise runs keep the seeds fixed). Delta F1: mean change from the reference run. Agreement: Jaccard overlap between the predicted synonym pairs of a noisy run and of the reference run. Seed draws (seeded clustering only, no noise, 10 draws): SD of F1, and mean agreement of draws 1 to 9 with draw 0. All noise levels are in Supplementary Table S3.

,Dataset,Technique,Model,F1 reference run,F1 (sigma = 0.30),SD (sigma = 0.30),Delta F1 (sigma = 0.30),Agreement (sigma = 0.30),SD over seed draws,Agreement between seed draws
0,ELSST,Clustering,All-MPNet-Base-v2,0.508,0.507,0.008,-0.001,0.818,,
1,ELSST,Clustering,MPNet-Personality,0.487,0.478,0.004,-0.009,0.819,,
2,ELSST,Clustering,SciBERT (SciVocab),0.327,0.304,0.029,-0.024,0.655,,
3,ELSST,Clustering,BERT Base,0.286,0.274,0.086,-0.012,0.548,,
4,ELSST,Pairwise,All-MPNet-Base-v2,0.483,0.382,0.002,-0.100,0.576,,
5,ELSST,Pairwise,MPNet-Personality,0.453,0.374,0.003,-0.079,0.629,,
6,ELSST,Pairwise,SciBERT (SciVocab),0.272,0.213,0.003,-0.059,0.305,,
7,ELSST,Pairwise,BERT Base,0.161,0.058,0.003,-0.102,0.108,,
8,ELSST,Seeded,All-MPNet-Base-v2,0.467,0.367,0.006,-0.100,0.529,0.012,0.677
9,ELSST,Seeded,MPNet-Personality,0.393,0.299,0.005,-0.093,0.547,0.008,0.764


### Table 5. Rare-word test

In [7]:
rare = tidy(read_results("rare_word.csv"))
bands = tidy(read_results("rare_word_by_overlap.csv"))
pooled = bands[bands["overlap_band"] == "pooled"][["dataset", "technique", "model", "delta_recall"]]
rare = rare.merge(pooled.rename(columns={"delta_recall": "delta_within"}), on=["dataset", "technique", "model"])
rec_common = exact_rate(rare["recall_common"], rare["n_common"])
rec_rare = exact_rate(rare["recall_rare"], rare["n_rare"])

table5 = pd.DataFrame({
    "Dataset": rare["dataset"], "Technique": rare["technique"], "Model": rare["model"],
    "Recall (common)": rec_common.map("{:.3f}".format), "Recall (rare)": rec_rare.map("{:.3f}".format),
})
table5["Delta recall"] = (rec_common - rec_rare).map("{:+.3f}".format)
table5["Delta recall within overlap bands"] = rare["delta_within"].map("{:+.3f}".format)
n_rare = rare.groupby("dataset")["n_rare"].first()
save_table(table5, "table5_rare_word",
           "Rare-word test. Recall on positive pairs in the top (common) and bottom (rare) decile of word frequency. "
           "Delta recall = recall (common) - recall (rare). The last column compares rare and common pairs within "
           "the same lexical-overlap band (Jaccard = 0, 0 to 0.5, > 0.5) and pools the bands; values near zero mean "
           "that the raw gap is explained by lexical overlap. "
           f"N rare = {n_rare['ELSST']:,} on ELSST and {n_rare['APA']:,} on APA. Bands: Supplementary Table S4.")

**table5_rare_word.** Rare-word test. Recall on positive pairs in the top (common) and bottom (rare) decile of word frequency. Delta recall = recall (common) - recall (rare). The last column compares rare and common pairs within the same lexical-overlap band (Jaccard = 0, 0 to 0.5, > 0.5) and pools the bands; values near zero mean that the raw gap is explained by lexical overlap. N rare = 156 on ELSST and 567 on APA. Bands: Supplementary Table S4.

,Dataset,Technique,Model,Recall (common),Recall (rare),Delta recall,Delta recall within overlap bands
0,ELSST,Clustering,All-MPNet-Base-v2,0.462,0.410,+0.051,-0.137
1,ELSST,Clustering,MPNet-Personality,0.474,0.391,+0.083,-0.109
2,ELSST,Clustering,SciBERT (SciVocab),0.263,0.205,+0.058,-0.085
3,ELSST,Clustering,BERT Base,0.333,0.173,+0.160,+0.039
4,ELSST,Pairwise,All-MPNet-Base-v2,0.391,0.250,+0.141,-0.030
5,ELSST,Pairwise,MPNet-Personality,0.372,0.224,+0.147,-0.004
6,ELSST,Pairwise,SciBERT (SciVocab),0.192,0.128,+0.064,-0.033
7,ELSST,Pairwise,BERT Base,0.115,0.064,+0.051,+0.008
8,ELSST,Pairwise,Character 3-gram (baseline),0.263,0.141,+0.122,+0.023
9,ELSST,Seeded,All-MPNet-Base-v2,0.353,0.231,+0.122,-0.042


### Table 6. Structural alignment test

In [8]:
# Part A: one value per model and data set
rho = tidy(read_results("structural_alignment.csv"))
table6a = rho.pivot(index="model", columns="dataset", values="spearman_rho")
table6a = table6a.reindex([m for m in MODEL_ORDER if m in table6a.index])[list(DATASET_NAMES.values())]
n_pairs = rho.groupby("dataset")["n_pairs"].first()
table6a = fmt(table6a.reset_index().rename(columns={"model": "Model", "ELSST": "rho ELSST", "APA": "rho APA"}))
save_table(table6a, "table6a_structural_similarity",
           "Structural alignment, part A. Spearman rho between similarity (cosine; Jaccard for the baseline) and "
           "thesaurus path length among non-synonym pairs with a path (length >= 2). A negative rho means that "
           f"similarity falls as concepts are further apart. Pairs: {n_pairs['ELSST']:,} on ELSST and "
           f"{n_pairs['APA']:,} on APA (rho on a fixed random sample of 5 million pairs).")

**table6a_structural_similarity.** Structural alignment, part A. Spearman rho between similarity (cosine; Jaccard for the baseline) and thesaurus path length among non-synonym pairs with a path (length >= 2). A negative rho means that similarity falls as concepts are further apart. Pairs: 819,534 on ELSST and 59,534,803 on APA (rho on a fixed random sample of 5 million pairs).

dataset,Model,rho ELSST,rho APA
0,All-MPNet-Base-v2,-0.221,-0.249
1,MPNet-Personality,-0.134,-0.162
2,SciBERT (SciVocab),-0.150,-0.174
3,BERT Base,-0.180,-0.275
4,Character 3-gram (baseline),-0.029,-0.085


In [9]:
# Part B: false-merge rate by path length, per pipeline
merge = tidy(read_results("structural_false_merge_by_distance.csv"))
band_cols = [col for col in merge.columns if col.startswith("fpr_")]
table6b = merge[["dataset", "technique", "model"] + band_cols + ["near_miss_share"]].copy()
table6b = fmt(table6b, 4, band_cols)
table6b = fmt(table6b, 3, ["near_miss_share"])
table6b.columns = (["Dataset", "Technique", "Model"]
                   + ["Path " + col.replace("fpr_path_", "").replace("fpr_not_connected", "none") for col in band_cols]
                   + ["Near-miss share"])
save_table(table6b, "table6b_structural_false_merges",
           "Structural alignment, part B. False-positive rate of each pipeline by thesaurus path length between the "
           "two concepts (none: not connected). Near-miss share: fraction of all false positives that join concepts "
           "at path length 2 to 4.")

**table6b_structural_false_merges.** Structural alignment, part B. False-positive rate of each pipeline by thesaurus path length between the two concepts (none: not connected). Near-miss share: fraction of all false positives that join concepts at path length 2 to 4.

,Dataset,Technique,Model,Path 2,Path 3,Path 4,Path 5-6,Path 7+,Path none,Near-miss share
0,ELSST,Clustering,All-MPNet-Base-v2,0.0553,0.0097,0.0031,0.0012,0.0002,0.0002,0.421
1,ELSST,Clustering,MPNet-Personality,0.0437,0.0074,0.0032,0.0018,0.0003,0.0003,0.310
2,ELSST,Clustering,SciBERT (SciVocab),0.0437,0.0067,0.0034,0.0017,0.0002,0.0003,0.328
3,ELSST,Clustering,BERT Base,0.0496,0.0106,0.0036,0.0014,0.0004,0.0005,0.284
4,ELSST,Pairwise,All-MPNet-Base-v2,0.0193,0.0044,0.0008,0.0005,0.0001,0.0001,0.451
5,ELSST,Pairwise,MPNet-Personality,0.0196,0.0040,0.0012,0.0006,0.0001,0.0001,0.406
6,ELSST,Pairwise,SciBERT (SciVocab),0.0317,0.0059,0.0022,0.0011,0.0002,0.0003,0.302
7,ELSST,Pairwise,BERT Base,0.0111,0.0014,0.0007,0.0005,0.0002,0.0003,0.125
8,ELSST,Pairwise,Character 3-gram (baseline),0.0156,0.0029,0.0016,0.0006,0.0001,0.0001,0.303
9,ELSST,Seeded,All-MPNet-Base-v2,0.0162,0.0047,0.0004,0.0004,0.0001,0.0001,0.427


---
## 2. Supplementary tables

### S1. Pair counts

In [10]:
def n_rows(folder, file_name):
    path = folder / file_name
    return pq.ParquetFile(path).metadata.num_rows if path.exists() else np.nan

rows = []
for key, label in [("elsst_train", "ELSST train (calibration)"), ("elsst_test", "ELSST test"), ("apa", "APA (full)")]:
    folder, prefix = config.DATASETS[key]["folder"], config.DATASETS[key]["prefix"]
    rows.append({
        "Split": label,
        "Concepts": n_rows(folder, f"{prefix}concepts.parquet"),
        "Terms": n_rows(folder, f"{prefix}terms.parquet"),
        "Synonym pairs": n_rows(folder, f"{prefix}positive_pairs.parquet"),
        "Non-synonym pairs": n_rows(folder, f"{prefix}negative_pairs.parquet"),
        "Broader/narrower pairs": n_rows(folder, f"{prefix}hierarchy_pairs.parquet"),
    })
s1 = pd.DataFrame(rows)
for col in s1.columns[1:]:
    s1[col] = s1[col].map(lambda v: "" if pd.isna(v) else f"{int(v):,}")
save_table(s1, "S1_pair_counts",
           "Pair counts. Non-synonym pairs combine every label of one concept with every label of another concept, "
           "excluding concepts on the same family line (all ancestors and descendants). Broader/narrower pairs are "
           "kept apart from the benchmark (Supplementary Table S5).")

**S1_pair_counts.** Pair counts. Non-synonym pairs combine every label of one concept with every label of another concept, excluding concepts on the same family line (all ancestors and descendants). Broader/narrower pairs are kept apart from the benchmark (Supplementary Table S5).

,Split,Concepts,Terms,Synonym pairs,Non-synonym pairs,Broader/narrower pairs
0,ELSST train (calibration),"2,404","4,270","3,587","9,098,662","12,066"
1,ELSST test,"1,031","1,876","1,557","1,754,671","2,522"
2,APA (full),"7,333","11,014","5,639","60,562,625","80,327"


### S2. Calibration on the ELSST training split

In [11]:
cal_dir = config.CALIBRATION_DIR
rows = []

pw = pd.read_csv(cal_dir / "pairwise_results.csv")
for key, sub in pw.groupby("model"):
    best = sub.loc[sub["f1_mean"].idxmax()]
    rows.append({"Technique": "Pairwise", "Model": KEY_TO_NAME.get(key, key),
                 "Selected parameters": f"threshold = {best['threshold']:.2f}",
                 "Precision": best["precision_mean"], "Recall": best["recall_mean"], "F1": best["f1_mean"]})

cl = pd.read_csv(cal_dir / "clustering_results.csv")
for name, sub in cl.groupby("model"):
    best = sub.loc[sub["f1"].idxmax()]
    dims = "none" if best["n_components"] >= 768 else int(best["n_components"])
    rows.append({"Technique": "Clustering", "Model": KEY_TO_NAME.get(name, name),
                 "Selected parameters": f"UMAP dims = {dims}, min cluster size = {int(best['min_cluster_size'])}, "
                                        f"min samples = {int(best['min_samples'])}",
                 "Precision": best["precision"], "Recall": best["recall"], "F1": best["f1"]})

sd = pd.read_csv(cal_dir / "seed_cluster_results.csv")
sd_mean = sd.groupby(["model_name", "num_initial_seeds", "threshold"])[["precision", "recall", "f1"]].mean().reset_index()
for key, sub in sd_mean.groupby("model_name"):
    best = sub.loc[sub["f1"].idxmax()]
    rows.append({"Technique": "Seeded", "Model": KEY_TO_NAME.get(key, key),
                 "Selected parameters": f"seeds = {int(best['num_initial_seeds'])}, threshold = {best['threshold']:.2f}",
                 "Precision": best["precision"], "Recall": best["recall"], "F1": best["f1"]})

s2 = tidy(pd.DataFrame(rows).rename(columns={"Technique": "technique", "Model": "model"}))
s2 = fmt(s2.rename(columns={"technique": "Technique", "model": "Model"}))
save_table(s2, "S2_calibration",
           "Parameters selected by grid search on the ELSST training split (highest F1; seeded clustering: mean "
           "over trials) and the training-split scores. These values are fixed in scripts/config.py for all "
           "evaluations.")

**S2_calibration.** Parameters selected by grid search on the ELSST training split (highest F1; seeded clustering: mean over trials) and the training-split scores. These values are fixed in scripts/config.py for all evaluations.

,Technique,Model,Selected parameters,Precision,Recall,F1
0,Clustering,All-MPNet-Base-v2,"UMAP dims = none, min cluster size = 2, min sa...",0.462,0.418,0.439
1,Clustering,MPNet-Personality,"UMAP dims = none, min cluster size = 2, min sa...",0.434,0.392,0.412
2,Clustering,SciBERT (SciVocab),"UMAP dims = none, min cluster size = 2, min sa...",0.269,0.231,0.249
3,Clustering,BERT Base,"UMAP dims = none, min cluster size = 2, min sa...",0.302,0.213,0.250
4,Pairwise,All-MPNet-Base-v2,threshold = 0.74,0.532,0.386,0.447
5,Pairwise,MPNet-Personality,threshold = 0.71,0.470,0.364,0.411
6,Pairwise,SciBERT (SciVocab),threshold = 0.90,0.230,0.203,0.215
7,Pairwise,BERT Base,threshold = 0.91,0.123,0.108,0.115
8,Pairwise,Character 3-gram (baseline),threshold = 0.45,0.248,0.160,0.195
9,Seeded,All-MPNet-Base-v2,"seeds = 250, threshold = 0.75",0.516,0.357,0.422


### S3. Embedding stability at every noise level

In [12]:
s3 = stab[["dataset", "technique", "model", "noise_level", "variation_source", "n_runs",
           "mean_f1", "sd_f1", "mean_delta_f1", "mean_agreement", "min_agreement"]].copy()
s3 = fmt(s3, 3, ["mean_f1", "sd_f1", "mean_delta_f1", "mean_agreement", "min_agreement"])
s3.columns = ["Dataset", "Technique", "Model", "Sigma", "Variation", "Runs",
              "Mean F1", "SD F1", "Delta F1", "Mean agreement", "Min agreement"]
save_table(s3, "S3_stability_all_levels",
           "Embedding stability at every noise level. Agreement: Jaccard overlap of predicted synonym pairs with the "
           "noise-free run.")

**S3_stability_all_levels.** Embedding stability at every noise level. Agreement: Jaccard overlap of predicted synonym pairs with the noise-free run.

,Dataset,Technique,Model,Sigma,Variation,Runs,Mean F1,SD F1,Delta F1,Mean agreement,Min agreement
0,ELSST,Clustering,All-MPNet-Base-v2,0.00,none (deterministic),1,0.508,0.000,0.000,1.000,1.000
1,ELSST,Clustering,All-MPNet-Base-v2,0.05,embedding noise,10,0.503,0.005,-0.005,0.951,0.929
2,ELSST,Clustering,All-MPNet-Base-v2,0.10,embedding noise,10,0.505,0.005,-0.003,0.927,0.907
3,ELSST,Clustering,All-MPNet-Base-v2,0.20,embedding noise,10,0.502,0.007,-0.006,0.875,0.863
4,ELSST,Clustering,All-MPNet-Base-v2,0.30,embedding noise,10,0.507,0.008,-0.001,0.818,0.800
...,...,...,...,...,...,...,...,...,...,...,...
115,APA,Seeded,BERT Base,0.00,seed draws,10,0.026,0.004,-0.008,0.247,0.200
116,APA,Seeded,BERT Base,0.05,embedding noise,10,0.034,0.002,-0.000,0.600,0.495
117,APA,Seeded,BERT Base,0.10,embedding noise,10,0.030,0.001,-0.004,0.505,0.458
118,APA,Seeded,BERT Base,0.20,embedding noise,10,0.024,0.001,-0.010,0.304,0.265


### S4. Rare-word gap by lexical-overlap band

In [13]:
s4 = bands[bands["overlap_band"] != "pooled"].copy()
s4 = fmt(s4[["dataset", "technique", "model", "overlap_band", "n_rare", "n_common", "recall_rare", "recall_common"]],
         3, ["recall_rare", "recall_common"])
s4["delta"] = bands[bands["overlap_band"] != "pooled"]["delta_recall"].map("{:+.3f}".format).values
s4.columns = ["Dataset", "Technique", "Model", "Overlap band", "N rare", "N common",
              "Recall (rare)", "Recall (common)", "Delta recall"]
save_table(s4, "S4_rare_word_by_overlap",
           "Rare-word gap within lexical-overlap bands (character 3-gram Jaccard of the two terms). "
           "Delta recall = recall (common) - recall (rare).")

**S4_rare_word_by_overlap.** Rare-word gap within lexical-overlap bands (character 3-gram Jaccard of the two terms). Delta recall = recall (common) - recall (rare).

,Dataset,Technique,Model,Overlap band,N rare,N common,Recall (rare),Recall (common),Delta recall
0,ELSST,Clustering,All-MPNet-Base-v2,Jaccard = 0,92,32,0.196,0.062,-0.133
1,ELSST,Clustering,All-MPNet-Base-v2,0 < Jaccard ≤ 0.5,46,96,0.674,0.490,-0.184
2,ELSST,Clustering,All-MPNet-Base-v2,Jaccard > 0.5,18,28,0.833,0.821,-0.012
4,ELSST,Clustering,MPNet-Personality,Jaccard = 0,92,32,0.152,0.125,-0.027
5,ELSST,Clustering,MPNet-Personality,0 < Jaccard ≤ 0.5,46,96,0.696,0.490,-0.206
...,...,...,...,...,...,...,...,...,...
97,APA,Seeded,SciBERT (SciVocab),0 < Jaccard ≤ 0.5,268,332,0.138,0.220,+0.082
98,APA,Seeded,SciBERT (SciVocab),Jaccard > 0.5,85,154,0.412,0.338,-0.074
100,APA,Seeded,BERT Base,Jaccard = 0,214,79,0.000,0.038,+0.038
101,APA,Seeded,BERT Base,0 < Jaccard ≤ 0.5,268,332,0.060,0.148,+0.088


### S5. Broader/narrower pairs

In [14]:
hier = tidy(read_results("hierarchy_merge_rate.csv"))
s5 = fmt(hier[["dataset", "technique", "model", "merge_rate_depth_1", "merge_rate_depth_2",
               "merge_rate_depth_3+", "merge_rate_depth_all", "fpr_benchmark_negatives"]], 4,
         ["merge_rate_depth_1", "merge_rate_depth_2", "merge_rate_depth_3+", "merge_rate_depth_all",
          "fpr_benchmark_negatives"])
s5.columns = ["Dataset", "Technique", "Model", "Depth 1", "Depth 2", "Depth 3+", "All depths",
              "FPR benchmark negatives"]
counts = hier.groupby("dataset")[["n_depth_1", "n_depth_2", "n_depth_3+"]].first()
save_table(s5, "S5_broader_narrower",
           "Share of broader/narrower pairs predicted as synonyms, by number of broader steps between the two "
           "concepts (depth 1 = direct parent). These pairs are not part of the benchmark. Pairs at depth 1 / 2 / 3+: "
           + "; ".join(f"{ds} {int(r['n_depth_1']):,} / {int(r['n_depth_2']):,} / {int(r['n_depth_3+']):,}"
                       for ds, r in counts.iterrows()) + ".")

**S5_broader_narrower.** Share of broader/narrower pairs predicted as synonyms, by number of broader steps between the two concepts (depth 1 = direct parent). These pairs are not part of the benchmark. Pairs at depth 1 / 2 / 3+: APA 25,419 / 24,371 / 30,537; ELSST 986 / 684 / 852.

,Dataset,Technique,Model,Depth 1,Depth 2,Depth 3+,All depths,FPR benchmark negatives
0,ELSST,Clustering,All-MPNet-Base-v2,0.1572,0.0175,0.0035,0.0674,0.0004
1,ELSST,Clustering,MPNet-Personality,0.1329,0.0278,0.0059,0.0615,0.0005
2,ELSST,Clustering,SciBERT (SciVocab),0.0700,0.0073,0.0000,0.0293,0.0004
3,ELSST,Clustering,BERT Base,0.0822,0.0219,0.0023,0.0389,0.0006
4,ELSST,Pairwise,All-MPNet-Base-v2,0.0984,0.0058,0.0012,0.0404,0.0001
5,ELSST,Pairwise,MPNet-Personality,0.0842,0.0015,0.0012,0.0337,0.0002
6,ELSST,Pairwise,SciBERT (SciVocab),0.0517,0.0044,0.0012,0.0218,0.0004
7,ELSST,Pairwise,BERT Base,0.0274,0.0058,0.0000,0.0123,0.0003
8,ELSST,Pairwise,Character 3-gram (baseline),0.0588,0.0219,0.0153,0.0341,0.0002
9,ELSST,Seeded,All-MPNet-Base-v2,0.0923,0.0102,0.0012,0.0393,0.0001


### S6. Paired F1 differences

In [15]:
diffs = read_results("bootstrap_f1_differences.csv")
s6 = fmt(diffs[["dataset", "pipeline_a", "pipeline_b", "f1_difference", "ci_low", "ci_high",
                "share_samples_a_better"]])
s6.columns = ["Dataset", "Pipeline A", "Pipeline B", "F1 difference (A - B)", "CI low", "CI high",
              "Share of samples A better"]
save_table(s6, "S6_f1_differences",
           "Paired F1 differences between pipelines with 95% concept-level bootstrap intervals (same 1,000 samples "
           "for every pipeline). An interval that excludes 0 marks a difference beyond concept sampling variation.")

**S6_f1_differences.** Paired F1 differences between pipelines with 95% concept-level bootstrap intervals (same 1,000 samples for every pipeline). An interval that excludes 0 marks a difference beyond concept sampling variation.

,Dataset,Pipeline A,Pipeline B,F1 difference (A - B),CI low,CI high,Share of samples A better
0,ELSST,clustering | All-MPNet-Base-v2,clustering | MPNet-Personality,0.021,-0.017,0.059,0.859
1,ELSST,clustering | All-MPNet-Base-v2,clustering | SciBERT (SciVocab),0.181,0.136,0.226,1.000
2,ELSST,clustering | All-MPNet-Base-v2,clustering | BERT Base,0.222,0.175,0.269,1.000
3,ELSST,clustering | All-MPNet-Base-v2,pairwise | All-MPNet-Base-v2,0.025,-0.015,0.065,0.885
4,ELSST,clustering | All-MPNet-Base-v2,pairwise | MPNet-Personality,0.054,0.017,0.096,0.997
...,...,...,...,...,...,...,...
151,APA,seeded | All-MPNet-Base-v2,seeded | SciBERT (SciVocab),0.275,0.255,0.293,1.000
152,APA,seeded | All-MPNet-Base-v2,seeded | BERT Base,0.280,0.261,0.298,1.000
153,APA,seeded | MPNet-Personality,seeded | SciBERT (SciVocab),0.284,0.263,0.304,1.000
154,APA,seeded | MPNet-Personality,seeded | BERT Base,0.289,0.269,0.309,1.000


### S7. Ranking: top-k recall and correct matches among reviewed candidates

In [16]:
rank = tidy(read_results("topk_and_review.csv"))
k_cols = [f"recall_at_{k}" for k in config.TOPK_LIST]
n_cols = [f"correct_in_top_{n}" for n in config.REVIEW_BUDGETS]
s7 = fmt(rank[["dataset", "model"] + k_cols + n_cols], 3, k_cols)
s7.columns = (["Dataset", "Model"] + [f"Recall@{k}" for k in config.TOPK_LIST]
              + [f"Correct in top {n:,}" for n in config.REVIEW_BUDGETS])
save_table(s7, "S7_ranking",
           "Recall@k: share of synonym pairs (both directions) whose synonym is among the k nearest terms "
           "(ties: expected value under random order). Correct in top N: synonym pairs among the N most "
           "similar benchmark pairs, as a curator reviewing a ranked list would see them.")

**S7_ranking.** Recall@k: share of synonym pairs (both directions) whose synonym is among the k nearest terms (ties: expected value under random order). Correct in top N: synonym pairs among the N most similar benchmark pairs, as a curator reviewing a ranked list would see them.

,Dataset,Model,Recall@1,Recall@5,Recall@10,Correct in top 100,Correct in top 500,"Correct in top 1,000"
0,ELSST,All-MPNet-Base-v2,0.318,0.706,0.813,99,405,638
1,ELSST,MPNet-Personality,0.313,0.695,0.796,99,379,611
2,ELSST,SciBERT (SciVocab),0.209,0.442,0.519,85,256,349
3,ELSST,BERT Base,0.215,0.447,0.522,55,146,232
4,ELSST,Character 3-gram (baseline),0.161,0.375,0.473,72,205,293
5,APA,All-MPNet-Base-v2,0.313,0.593,0.677,94,443,791
6,APA,MPNet-Personality,0.306,0.576,0.663,85,415,737
7,APA,SciBERT (SciVocab),0.151,0.317,0.379,26,89,151
8,APA,BERT Base,0.146,0.310,0.372,47,114,150
9,APA,Character 3-gram (baseline),0.197,0.351,0.414,64,367,630


### S8. F1 with preferred-label negatives only

In [17]:
s8 = fmt(perf[["dataset", "technique", "model", "f1", "f1_pref_pref_only"]])
s8.columns = ["Dataset", "Technique", "Model", "F1 (all-label negatives)", "F1 (preferred-label negatives only)"]
save_table(s8, "S8_preferred_label_negatives",
           "F1 with all-label non-synonym pairs (main analysis) and with non-synonym pairs between preferred labels "
           "only, as in the submitted version.")

**S8_preferred_label_negatives.** F1 with all-label non-synonym pairs (main analysis) and with non-synonym pairs between preferred labels only, as in the submitted version.

,Dataset,Technique,Model,F1 (all-label negatives),F1 (preferred-label negatives only)
0,ELSST,Clustering,All-MPNet-Base-v2,0.508,0.586
1,ELSST,Clustering,MPNet-Personality,0.487,0.572
2,ELSST,Clustering,SciBERT (SciVocab),0.327,0.395
3,ELSST,Clustering,BERT Base,0.286,0.357
4,ELSST,Pairwise,All-MPNet-Base-v2,0.483,0.514
5,ELSST,Pairwise,MPNet-Personality,0.453,0.485
6,ELSST,Pairwise,SciBERT (SciVocab),0.272,0.319
7,ELSST,Pairwise,BERT Base,0.161,0.181
8,ELSST,Pairwise,Character 3-gram (baseline),0.203,0.222
9,ELSST,Seeded,All-MPNet-Base-v2,0.451,0.481


### S9. Seeded clustering over seed draws

In [18]:
draws = read_results("seeded_draws.csv")
draws["model"] = draws["model"].map(lambda m: KEY_TO_NAME.get(m, m))
s9 = draws.groupby(["dataset", "model"])["f1"].agg(["mean", "std", "min", "max"]).reset_index()
s9 = fmt(tidy(s9))
s9.columns = ["Dataset", "Model", "Mean F1", "SD F1", "Min F1", "Max F1"]
save_table(s9, "S9_seed_draws",
           f"Seeded clustering over {config.N_SEED_DRAWS} random seed draws (seeds sampled uniformly from all terms, "
           "no gold information).")

**S9_seed_draws.** Seeded clustering over 10 random seed draws (seeds sampled uniformly from all terms, no gold information).

,Dataset,Model,Mean F1,SD F1,Min F1,Max F1
0,ELSST,All-MPNet-Base-v2,0.451,0.012,0.424,0.467
1,ELSST,MPNet-Personality,0.390,0.008,0.377,0.406
2,ELSST,SciBERT (SciVocab),0.209,0.020,0.181,0.231
3,ELSST,BERT Base,0.159,0.015,0.139,0.184
4,APA,All-MPNet-Base-v2,0.310,0.009,0.299,0.322
5,APA,MPNet-Personality,0.315,0.006,0.306,0.322
6,APA,SciBERT (SciVocab),0.029,0.004,0.023,0.034
7,APA,BERT Base,0.026,0.004,0.019,0.034


### S10. Mean similarity by path length

In [19]:
sim = tidy(read_results("structural_similarity_by_distance.csv"))
s10 = sim.pivot_table(index=["dataset", "model"], columns="path_length", values="mean_similarity", observed=True)
s10 = fmt(tidy(s10.reset_index()))
s10 = s10.rename(columns={"dataset": "Dataset", "model": "Model"})
save_table(s10, "S10_similarity_by_path_length",
           "Mean similarity of non-synonym pairs by thesaurus path length (cosine; Jaccard for the baseline).")

**S10_similarity_by_path_length.** Mean similarity of non-synonym pairs by thesaurus path length (cosine; Jaccard for the baseline).

path_length,Dataset,Model,2,3,4,5-6,7+
0,ELSST,All-MPNet-Base-v2,0.422,0.326,0.272,0.223,0.152
1,ELSST,MPNet-Personality,0.342,0.245,0.201,0.164,0.118
2,ELSST,SciBERT (SciVocab),0.761,0.732,0.721,0.705,0.682
3,ELSST,BERT Base,0.703,0.665,0.649,0.640,0.606
4,ELSST,Character 3-gram (baseline),0.090,0.036,0.027,0.020,0.012
5,APA,All-MPNet-Base-v2,0.368,0.290,0.238,0.180,0.124
6,APA,MPNet-Personality,0.292,0.222,0.179,0.138,0.109
7,APA,SciBERT (SciVocab),0.747,0.719,0.705,0.688,0.667
8,APA,BERT Base,0.672,0.640,0.624,0.605,0.561
9,APA,Character 3-gram (baseline),0.078,0.035,0.021,0.015,0.010


---
## 3. Files written

In [20]:
for path in sorted(OUT_DIR.glob("*.csv")):
    print(path.name)

S10_similarity_by_path_length.csv
S1_pair_counts.csv
S2_calibration.csv
S3_stability_all_levels.csv
S4_rare_word_by_overlap.csv
S5_broader_narrower.csv
S6_f1_differences.csv
S7_ranking.csv
S8_preferred_label_negatives.csv
S9_seed_draws.csv
table1_primary_performance.csv
table2_lexical_trap.csv
table3_semantic_decay.csv
table4_embedding_stability.csv
table5_rare_word.csv
table6a_structural_similarity.csv
table6b_structural_false_merges.csv
